# Feature Engineering
Creating new features and preparing the data for modeling.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

df_train = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')

df_train['dataset'] = 'train'
df_test['dataset'] = 'test'
combined = pd.concat([df_train, df_test], ignore_index=True)

print('Combined shape:', combined.shape)

Combined shape: (1309, 13)


In [2]:
# Filling missing values

# Age - filling with median per Pclass and Sex, because it is more accurate than regular median
age_medians = combined.groupby(['Pclass', 'Sex'])['Age'].transform('median')
combined['Age'] = combined['Age'].fillna(age_medians)

# Embarked - filling with most common value
combined['Embarked'] = combined['Embarked'].fillna(combined['Embarked'].mode()[0])

# Fare - filling with median of same Pclass
combined['Fare'] = combined.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))

# Check if all values were filled
print('Missing values after filling:')
print(combined[['Age', 'Embarked', 'Fare']].isnull().sum())

Missing values after filling:
Age         0
Embarked    0
Fare        0
dtype: int64


In [3]:
# Calculating new statistics

# Family size
combined['FamilySize'] = combined['SibSp'] + combined['Parch'] + 1

# Is alone?
combined['IsAlone'] = (combined['FamilySize'] == 1).astype(int)

# Title from Name
combined['Title'] = combined['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Group rare titles
rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr',
               'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
combined['Title'] = combined['Title'].replace(rare_titles, 'Rare')
combined['Title'] = combined['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

print('Title value counts:')
print(combined['Title'].value_counts())

# Age bands
combined['AgeBand'] = pd.cut(combined['Age'], bins=[0, 12, 18, 35, 60, 100],
                             labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'])

# Fare bands
combined['LogFare'] = np.log1p(combined['Fare'])

combined[['Age', 'AgeBand', 'Fare', 'LogFare', 'FamilySize', 'IsAlone', 'Title']].head(10)

Title value counts:
Title
Mr        757
Miss      264
Mrs       198
Master     61
Rare       29
Name: count, dtype: int64


,Age,AgeBand,Fare,LogFare,FamilySize,IsAlone,Title
0,22.0,YoungAdult,7.2500,2.110213,2,0,Mr
1,38.0,Adult,71.2833,4.280593,2,0,Mrs
2,26.0,YoungAdult,7.9250,2.188856,1,1,Miss
3,35.0,YoungAdult,53.1000,3.990834,2,0,Mrs
4,35.0,YoungAdult,8.0500,2.202765,1,1,Mr
5,25.0,YoungAdult,8.4583,2.246893,1,1,Mr
6,54.0,Adult,51.8625,3.967694,1,1,Mr
7,2.0,Child,21.0750,3.094446,5,0,Master
8,27.0,YoungAdult,11.1333,2.495954,3,0,Mrs
9,14.0,Teen,30.0708,3.436268,2,0,Mrs


In [4]:
# Label Encoding

le = LabelEncoder()
combined['Sex_encoded'] = le.fit_transform(combined['Sex'])
combined['Embarked_encoded'] = le.fit_transform(combined['Embarked'])
combined['Title_encoded'] = le.fit_transform(combined['Title'])
combined['AgeBand_encoded'] = le.fit_transform(combined['AgeBand'].astype(str))

FEATURES = ['Pclass', 'Sex_encoded', 'Age', 'FamilySize', 'IsAlone',
            'LogFare', 'Embarked_encoded', 'Title_encoded', 'AgeBand_encoded']

print('Features selected:', FEATURES)

train_processed = combined[combined['dataset'] == 'train'].copy()
test_processed = combined[combined['dataset'] == 'test'].copy()

train_processed.to_csv('../data/train_processed.csv', index=False)
test_processed.to_csv('../data/test_processed.csv', index=False)

print('\nSaved processed data')

Features selected: ['Pclass', 'Sex_encoded', 'Age', 'FamilySize', 'IsAlone', 'LogFare', 'Embarked_encoded', 'Title_encoded', 'AgeBand_encoded']

Saved processed data
